# 第92章 决策树

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 7 / 34 步：扩展监督/无监督模型工具箱**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** K近邻模型（KNN）  →  **本章任务：** 决策树  →  **下一步：** 随机森林
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

**背景引入**：我们拿到一堆特征想判断「值不值得买」「会不会流失」这类问题时，决策树会自动挑出最有区分力的字段，一层层把样本分开，还能把分裂规则翻译成能读懂的文字。它不需要对特征做缩放，训练完可以直接解释「为什么分成这一组」，很适合在建模早期快速建立一条可解释的基线。


## 本章目标

学完本章，你将能够：

- **理解**：理解「决策树」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「决策树」的关键输出指标。
- **迁移**：能把「决策树」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 核心概念

**背景引入**：面对一份有类别标签的数据，最直觉的做法是“按问题一步步问下去”——是不是会员？金额够不够？决策树正是把这样的判断规则画成一棵可解释的树。它无需给特征做缩放，天然适合带缺失和类别混杂的数据；但树太深就会把训练噪声也背下来，所以控制复杂度是它的核心功课。


- 树模型无需特征缩放
- 深树容易记住训练噪声（打个比方：树长得太深，等于把练习册答案都背下来，换个问法就不会做；浅一点、留点余地的树反而更应付得了没见过的题。）
- max_depth 与 min_samples_leaf 控制复杂度
- 内置 impurity importance 偏向可分裂机会多的特征


## 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 从浅树开始 | `tree.score()`、`.fit()`、`list()` | Iris 浅树便于展示规则和决策路径。 | 只看规则易读就认为结论因果可靠 |
| 复杂度与重要性 | `rows.append()`、`m.get_depth()`、`m.get_n_leaves()`、`m.score()` | 比较不同深度并查看模型采用的分裂特征。 | 让树无限生长后报告训练得分 |


## 例 1｜从浅树开始

Iris 浅树便于展示规则和决策路径。


<!-- math-foundation:chapter-92 -->
### 数学推导｜决策树用纯度下降选择切分

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜计算父节点不纯度。** 分类节点中类别比例为 $p_k$，Gini 为 $1-\sum_kp_k^2$。

**第 2 步｜候选切分产生左右子节点。** 切分后的期望不纯度是按样本量加权平均：

$$
I_{after}=\frac{n_L}{n}I(L)+\frac{n_R}{n}I(R)
$$

**第 3 步｜选择下降最多的切分。** $Gain=I(S)-I_{after}$。不断最大化训练集上的下降会生成很深的树，因此还需要深度、叶节点样本数或剪枝约束。

**把上面的关系收束为本章计算式：**

$$
Gini(S)=1-\sum_kp_k^2,\qquad Gain=I(S)-\frac{n_L}{n}I(L)-\frac{n_R}{n}I(R)
$$

**符号解释：** $p_k$ 是节点内类别 $k$ 的比例，$I$ 可取 Gini 或熵。

**代码对应：** 限制 `max_depth`、`min_samples_leaf` 并比较验证表现。

**使用边界：** 深树很容易记住训练噪声；特征重要性不等于因果贡献。


In [ ]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

data = load_iris(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=81
)
tree = DecisionTreeClassifier(
    max_depth=3, min_samples_leaf=4, random_state=81
).fit(X_train, y_train)
print(export_text(tree, feature_names=list(X.columns)))
print(
    "训练/测试:",
    round(tree.score(X_train, y_train), 3),
    round(tree.score(X_test, y_test), 3),
)


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：改一个参数观察变化。把上一格示例里的 `DecisionTreeClassifier(max_depth=3, min_samples_leaf=4, random_state=81)` 中的 `max_depth` 改成 `1` 再训练，观察叶节点数和训练/测试得分如何变化。填好后再回答：`max_depth` 越大时，训练得分一般会更高还是更低？


In [ ]:
try:
    pass
    # 请在下方填写代码：复制上方示例，把 max_depth 改成 1 后重新训练

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜复杂度与重要性

比较不同深度并查看模型采用的分裂特征。


In [ ]:
rows = []
for depth in [1, 2, 3, 5, None]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=81).fit(
        X_train, y_train
    )
    rows.append(
        [
            str(depth),
            m.get_depth(),
            m.get_n_leaves(),
            m.score(X_train, y_train),
            m.score(X_test, y_test),
        ]
    )
display(
    pd.DataFrame(
        rows, columns=["max_depth", "实际深度", "叶数", "train", "test"]
    )
)
display(
    pd.Series(tree.feature_importances_, index=X.columns).sort_values(
        ascending=False
    )
)


## 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # TODO: 在此粘贴或改写最接近的示例。
    # 记录：我改了什么？预期会发生什么？实际观察到什么？
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(
        {"修改": change_note, "预期": expected_change, "观察": observed_change}
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, model.predict(X)), 2))


### 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = model.predict(X_changed)
print("原始前2个预测：", np.round(model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print("预测变化：", np.round(changed_prediction[:2] - model.predict(X[:2]), 2))


### 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

_demo_data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in _demo_data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 只看规则易读就认为结论因果可靠
- 让树无限生长后报告训练得分
- 把 impurity importance 当成稳定真相
- 小样本下忽略树结构的不稳定性


## 练习与作业

1. 训练 min_samples_leaf 为 1、5、10 的树
2. 固定 max_depth=4
3. 比较叶节点数和测试准确率

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 92.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“训练 min_samples_leaf 为 1、5、10 的树”。
2. **独立完成**：不复制示例代码，完成“固定 max_depth=4”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“比较叶节点数和测试准确率”，用一两句话说明你修改了什么。

### 92.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 92.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


## 小结

使用决策树学习可解释的分裂规则，并通过深度、叶节点样本量和剪枝限制过拟合。


### 你已经掌握

- 训练 DecisionTreeClassifier
- 解释节点分裂和叶节点
- 比较训练与测试性能
- 读取特征重要性和文本规则


### 需要注意

- 只看规则易读就认为结论因果可靠
- 让树无限生长后报告训练得分
- 把 impurity importance 当成稳定真相
- 小样本下忽略树结构的不稳定性


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
# 完整答案：max_depth=1 的更浅决策树
tree_shallow = DecisionTreeClassifier(
    max_depth=1, min_samples_leaf=4, random_state=81
).fit(X_train, y_train)
print(
    "训练/测试:",
    round(tree_shallow.score(X_train, y_train), 3),
    round(tree_shallow.score(X_test, y_test), 3),
)


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
practice_rows = []
for leaf in [1, 5, 10]:
    m = DecisionTreeClassifier(
        max_depth=4, min_samples_leaf=leaf, random_state=81
    ).fit(X_train, y_train)
    practice_rows.append([leaf, m.get_n_leaves(), m.score(X_test, y_test)])
practice_result = pd.DataFrame(
    practice_rows, columns=["min_leaf", "leaves", "score"]
)
display(practice_result)
